# 02: TMDB Preprocessing

Builds seven candidate TMDB DataFrames from `01_overview_cleaning.ipynb`'s cleaned output (`tmdb_clean.parquet`), varying how missing `budget`/`revenue` and low `vote_count` rows are handled -- the `vote_count` filtering decision was deliberately deferred there (see 01's Section 3) pending this comparison. Also explores `keywords` co-occurrence to inform later list-field feature engineering. This is a scoping/exploration notebook: no candidate is selected yet and no output file is saved here.

## Setup

In [ ]:
%matplotlib inline

import sys
from pathlib import Path

import pandas as pd
from IPython.display import display
import numpy as np
from scipy.stats import kruskal

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src import config
from src.eda.helper import missingness_report
from src.report import log_findings

PROCESSED_DIR = config.PROCESSED_DIR

## Load cleaned dataset

In [ ]:
tmdb = pd.read_parquet(PROCESSED_DIR / "tmdb_clean.parquet")
print(f"Loaded tmdb_clean.parquet: {tmdb.shape}")

## 1. Candidate dataset construction

Seven variants of the cleaned dataset, crossing two design choices: whether rows missing `budget`/`revenue` are dropped or the columns are dropped entirely, and what minimum `vote_count` threshold to require (0, 5, or 10 -- plus a strict all-nulls-dropped baseline). Comparing their size and missingness helps decide which variant to carry into modeling.

### Build the 7 candidate DataFrames

In [ ]:
tmdb_no_nulls = tmdb.dropna(how="any")

tmdb_budget_revenue_gt0 = tmdb.dropna(subset=["budget", "revenue"])
tmdb_budget_revenue_gt0 = tmdb_budget_revenue_gt0[tmdb_budget_revenue_gt0["vote_count"] > 0]

tmdb_budget_revenue_gt5 = tmdb.dropna(subset=["budget", "revenue"])
tmdb_budget_revenue_gt5 = tmdb_budget_revenue_gt5[tmdb_budget_revenue_gt5["vote_count"] > 5]

tmdb_budget_revenue_gt10 = tmdb.dropna(subset=["budget", "revenue"])
tmdb_budget_revenue_gt10 = tmdb_budget_revenue_gt10[tmdb_budget_revenue_gt10["vote_count"] > 10]

tmdb_no_budget_revenue_gt0 = tmdb.drop(columns=["budget", "revenue"])
tmdb_no_budget_revenue_gt0 = tmdb_no_budget_revenue_gt0[tmdb_no_budget_revenue_gt0["vote_count"] > 0]

tmdb_no_budget_revenue_gt5 = tmdb.drop(columns=["budget", "revenue"])
tmdb_no_budget_revenue_gt5 = tmdb_no_budget_revenue_gt5[tmdb_no_budget_revenue_gt5["vote_count"] > 5]

tmdb_no_budget_revenue_gt10 = tmdb.drop(columns=["budget", "revenue"])
tmdb_no_budget_revenue_gt10 = tmdb_no_budget_revenue_gt10[tmdb_no_budget_revenue_gt10["vote_count"] > 10]

### Shapes of all 7 candidates

In [ ]:
tmdb_candidates = {
    "tmdb_no_nulls": tmdb_no_nulls,
    "tmdb_budget_revenue_gt0": tmdb_budget_revenue_gt0,
    "tmdb_budget_revenue_gt5": tmdb_budget_revenue_gt5,
    "tmdb_budget_revenue_gt10": tmdb_budget_revenue_gt10,
    "tmdb_no_budget_revenue_gt0": tmdb_no_budget_revenue_gt0,
    "tmdb_no_budget_revenue_gt5": tmdb_no_budget_revenue_gt5,
    "tmdb_no_budget_revenue_gt10": tmdb_no_budget_revenue_gt10,
}
for name, df in tmdb_candidates.items():
    print(f"{name}: {df.shape}")

### Missingness comparison across candidates

In [ ]:
missing_by_candidate = pd.concat(
    {name: missingness_report(df) for name, df in tmdb_candidates.items()},
    names=["candidate", "feature"],
).reset_index()

In [ ]:
# Pivoted view: one row per feature, one column per candidate -- makes it
# straightforward to compare a given feature's missingness across all 7
# candidates side by side, instead of scrolling through 7 separate tables.
missing_pct_by_feature = missing_by_candidate.pivot(
    index="feature", columns="candidate", values="pct_missing"
).round(2)
missing_pct_by_feature = missing_pct_by_feature.loc[
    missing_pct_by_feature.mean(axis=1).sort_values(ascending=False).index
]
missing_pct_by_feature


## 2. Tagline presence by language

Checks whether missing `tagline` values are evenly spread across languages or concentrated in specific ones -- if presence is skewed, dropping or imputing taglines could introduce language-correlated bias. Runs on the pre-filter `tmdb` DataFrame (not any of the 7 candidates); diagnostic only, no rows or columns are changed here.

### Tagline presence rate by top 10 languages for `tmdb_budget_revenue_gt0`

In [ ]:
tmdb_budget_revenue_gt0["has_tagline"] = tmdb_budget_revenue_gt0["tagline"].notna()

top10_languages = tmdb_budget_revenue_gt0["original_language"].value_counts().head(10).index

tagline_rate_by_language = (
    tmdb_budget_revenue_gt0[tmdb_budget_revenue_gt0["original_language"].isin(top10_languages)]
    .groupby("original_language")["has_tagline"]
    .mean()
    .sort_values()
)
print("Tagline presence rate by language (top 10 languages by row count):")
print(tagline_rate_by_language)

overall_has_tagline_rate = tmdb_budget_revenue_gt0["has_tagline"].mean()
overall_has_tagline_rate_br_gt0 = overall_has_tagline_rate
print(f"\nOverall tagline presence rate (all languages): {overall_has_tagline_rate:.3f}")

### Tagline presence rate by top 10 languages for `tmdb_no_budget_revenue_gt5`

In [ ]:
tmdb_no_budget_revenue_gt5["has_tagline"] = tmdb_no_budget_revenue_gt5["tagline"].notna()

top10_languages = tmdb_no_budget_revenue_gt5["original_language"].value_counts().head(10).index

tagline_rate_by_language = (
    tmdb_no_budget_revenue_gt5[tmdb_no_budget_revenue_gt5["original_language"].isin(top10_languages)]
    .groupby("original_language")["has_tagline"]
    .mean()
    .sort_values()
)
print("Tagline presence rate by language (top 10 languages by row count):")
print(tagline_rate_by_language)

overall_has_tagline_rate = tmdb_no_budget_revenue_gt5["has_tagline"].mean()
overall_has_tagline_rate_nbr_gt5 = overall_has_tagline_rate
print(f"\nOverall tagline presence rate (all languages): {overall_has_tagline_rate:.3f}")

### Drop `tagline`/`has_tagline`

In [ ]:
tmdb_budget_revenue_gt0 = tmdb_budget_revenue_gt0.drop(columns=["tagline", "has_tagline"])
tmdb_no_budget_revenue_gt5 = tmdb_no_budget_revenue_gt5.drop(columns=["tagline", "has_tagline"])

print(f"tmdb_budget_revenue_gt0: {tmdb_budget_revenue_gt0.shape}")
print(f"tmdb_no_budget_revenue_gt5: {tmdb_no_budget_revenue_gt5.shape}")

## 3. Overview presence by language

### Overview presence rate by top 10 languages for `tmdb_budget_revenue_gt0`

In [ ]:
tmdb_budget_revenue_gt0["has_overview"] = tmdb_budget_revenue_gt0["overview"].notna()

top10_languages = tmdb_budget_revenue_gt0["original_language"].value_counts().head(10).index

overview_rate_by_language = (
    tmdb_budget_revenue_gt0[tmdb_budget_revenue_gt0["original_language"].isin(top10_languages)]
    .groupby("original_language")["has_overview"]
    .mean()
    .sort_values()
)
print("Overview presence rate by language (top 10 languages by row count):")
print(overview_rate_by_language)

overall_has_overview_rate = tmdb_budget_revenue_gt0["has_overview"].mean()
overall_has_overview_rate_br_gt0 = overall_has_overview_rate
print(f"\nOverall overview presence rate (all languages): {overall_has_overview_rate:.3f}")

### Overview presence rate by top 10 languages for `tmdb_no_budget_revenue_gt5`

In [ ]:
tmdb_no_budget_revenue_gt5["has_overview"] = tmdb_no_budget_revenue_gt5["overview"].notna()

top10_languages = tmdb_no_budget_revenue_gt5["original_language"].value_counts().head(10).index

overview_rate_by_language = (
    tmdb_no_budget_revenue_gt5[tmdb_no_budget_revenue_gt5["original_language"].isin(top10_languages)]
    .groupby("original_language")["has_overview"]
    .mean()
    .sort_values()
)
print("Overview presence rate by language (top 10 languages by row count):")
print(overview_rate_by_language)

overall_has_overview_rate = tmdb_no_budget_revenue_gt5["has_overview"].mean()
overall_has_overview_rate_nbr_gt5 = overall_has_overview_rate
print(f"\nOverall overview presence rate (all languages): {overall_has_overview_rate:.3f}")

### Drop the has_overview feature

In [ ]:
if "has_overview" in tmdb_budget_revenue_gt0.columns:
    tmdb_budget_revenue_gt0 = tmdb_budget_revenue_gt0.drop(columns=["has_overview"])

if "has_overview" in tmdb_no_budget_revenue_gt5.columns:
    tmdb_no_budget_revenue_gt5 = tmdb_no_budget_revenue_gt5.drop(columns=["has_overview"])

## 4. Refine the chosen candidates

Continues with the two candidates carried forward from Section 1 -- `tmdb_budget_revenue_gt0` and `tmdb_no_budget_revenue_gt5`. Adds a presence flag (mirroring `has_tagline` above) for each of the five list-valued fields (`genres`, `keywords`, `production_companies`, `production_countries`, `spoken_languages`), then drops rows with a missing `release_date`.

### Add presence flags and drop rows missing `release_date`

In [ ]:
tmdb_budget_revenue_gt0 = tmdb_budget_revenue_gt0.copy()
tmdb_budget_revenue_gt0["has_genres"] = tmdb_budget_revenue_gt0["genres"].notna()
tmdb_budget_revenue_gt0["has_keywords"] = tmdb_budget_revenue_gt0["keywords"].notna()
tmdb_budget_revenue_gt0["has_production_companies"] = tmdb_budget_revenue_gt0["production_companies"].notna()
tmdb_budget_revenue_gt0["has_production_countries"] = tmdb_budget_revenue_gt0["production_countries"].notna()
tmdb_budget_revenue_gt0["has_spoken_languages"] = tmdb_budget_revenue_gt0["spoken_languages"].notna()
shape_before_release_date_drop = tmdb_budget_revenue_gt0.shape
tmdb_budget_revenue_gt0 = tmdb_budget_revenue_gt0[tmdb_budget_revenue_gt0["release_date"].notna()]
n_dropped_release_date_br_gt0 = shape_before_release_date_drop[0] - tmdb_budget_revenue_gt0.shape[0]
print(f"tmdb_budget_revenue_gt0: {tmdb_budget_revenue_gt0.shape} (dropped {n_dropped_release_date_br_gt0} missing release_date)")

In [ ]:
tmdb_no_budget_revenue_gt5 = tmdb_no_budget_revenue_gt5.copy()
tmdb_no_budget_revenue_gt5["has_genres"] = tmdb_no_budget_revenue_gt5["genres"].notna()
tmdb_no_budget_revenue_gt5["has_keywords"] = tmdb_no_budget_revenue_gt5["keywords"].notna()
tmdb_no_budget_revenue_gt5["has_production_companies"] = tmdb_no_budget_revenue_gt5["production_companies"].notna()
tmdb_no_budget_revenue_gt5["has_production_countries"] = tmdb_no_budget_revenue_gt5["production_countries"].notna()
tmdb_no_budget_revenue_gt5["has_spoken_languages"] = tmdb_no_budget_revenue_gt5["spoken_languages"].notna()
shape_before_release_date_drop = tmdb_no_budget_revenue_gt5.shape
tmdb_no_budget_revenue_gt5 = tmdb_no_budget_revenue_gt5[tmdb_no_budget_revenue_gt5["release_date"].notna()]
n_dropped_release_date_nbr_gt5 = shape_before_release_date_drop[0] - tmdb_no_budget_revenue_gt5.shape[0]
print(f"tmdb_no_budget_revenue_gt5: {tmdb_no_budget_revenue_gt5.shape} (dropped {n_dropped_release_date_nbr_gt5} missing release_date)")

### Add `has_overview` flag, then fill missing `overview` with an empty string

In [ ]:
tmdb_budget_revenue_gt0["has_overview"] = tmdb_budget_revenue_gt0["overview"].notna()
tmdb_budget_revenue_gt0["overview"] = tmdb_budget_revenue_gt0["overview"].fillna("")

tmdb_no_budget_revenue_gt5["has_overview"] = tmdb_no_budget_revenue_gt5["overview"].notna()
tmdb_no_budget_revenue_gt5["overview"] = tmdb_no_budget_revenue_gt5["overview"].fillna("")

## 5. Release date vs. language

### Decompose `release_date`

In [ ]:
tmdb_budget_revenue_gt0["release_year"] = tmdb_budget_revenue_gt0["release_date"].dt.year
tmdb_budget_revenue_gt0["release_month"] = tmdb_budget_revenue_gt0["release_date"].dt.month
tmdb_budget_revenue_gt0 = tmdb_budget_revenue_gt0.drop(columns=["release_date"])

In [ ]:
tmdb_no_budget_revenue_gt5["release_year"] = tmdb_no_budget_revenue_gt5["release_date"].dt.year
tmdb_no_budget_revenue_gt5["release_month"] = tmdb_no_budget_revenue_gt5["release_date"].dt.month
tmdb_no_budget_revenue_gt5 = tmdb_no_budget_revenue_gt5.drop(columns=["release_date"])


### Cyclical `release_month` & `release_year` significance for `tmdb_budget_revenue_gt0`

In [ ]:
tmdb_budget_revenue_gt0["release_month_sin"] = np.sin(2 * np.pi * tmdb_budget_revenue_gt0["release_month"] / 12)
tmdb_budget_revenue_gt0["release_month_cos"] = np.cos(2 * np.pi * tmdb_budget_revenue_gt0["release_month"] / 12)

top10_languages = tmdb_budget_revenue_gt0["original_language"].value_counts().head(10).index
top10_mask = tmdb_budget_revenue_gt0["original_language"].isin(top10_languages)

release_by_language = (
    tmdb_budget_revenue_gt0[top10_mask]
    .groupby("original_language")[["release_year", "release_month_sin", "release_month_cos"]]
    .mean()
    .sort_values("release_year")
)
release_by_language.loc["Overall (all languages)"] = tmdb_budget_revenue_gt0[
    ["release_year", "release_month_sin", "release_month_cos"]
].mean()

print("Mean release_year / release_month_sin / release_month_cos by language (top 10 languages by row count):")
display(release_by_language.round(3))

release_year_groups = [
    tmdb_budget_revenue_gt0.loc[tmdb_budget_revenue_gt0["original_language"] == lang, "release_year"].dropna()
    for lang in top10_languages
]
h_stat, p_value = kruskal(*release_year_groups)
h_stat_br_gt0, p_value_br_gt0 = h_stat, p_value
print(f"\nKruskal-Wallis test on release_year across top 10 languages: H={h_stat:.2f}, p={p_value:.4g}")

tmdb_budget_revenue_gt0 = tmdb_budget_revenue_gt0.drop(columns=["release_month"])

### Cyclical `release_month` & `release_year` significance for `tmdb_no_budget_revenue_gt5`

In [ ]:
tmdb_no_budget_revenue_gt5["release_month_sin"] = np.sin(2 * np.pi * tmdb_no_budget_revenue_gt5["release_month"] / 12)
tmdb_no_budget_revenue_gt5["release_month_cos"] = np.cos(2 * np.pi * tmdb_no_budget_revenue_gt5["release_month"] / 12)

top10_languages = tmdb_no_budget_revenue_gt5["original_language"].value_counts().head(10).index
top10_mask = tmdb_no_budget_revenue_gt5["original_language"].isin(top10_languages)

release_by_language = (
    tmdb_no_budget_revenue_gt5[top10_mask]
    .groupby("original_language")[["release_year", "release_month_sin", "release_month_cos"]]
    .mean()
    .sort_values("release_year")
)
release_by_language.loc["Overall (all languages)"] = tmdb_no_budget_revenue_gt5[
    ["release_year", "release_month_sin", "release_month_cos"]
].mean()

print("Mean release_year / release_month_sin / release_month_cos by language (top 10 languages by row count):")
display(release_by_language.round(3))

release_year_groups = [
    tmdb_no_budget_revenue_gt5.loc[tmdb_no_budget_revenue_gt5["original_language"] == lang, "release_year"].dropna()
    for lang in top10_languages
]
h_stat, p_value = kruskal(*release_year_groups)
h_stat_nbr_gt5, p_value_nbr_gt5 = h_stat, p_value
print(f"\nKruskal-Wallis test on release_year across top 10 languages: H={h_stat:.2f}, p={p_value:.4g}")

tmdb_no_budget_revenue_gt5 = tmdb_no_budget_revenue_gt5.drop(columns=["release_month"])

## 6. Title vs original_title equality

Checks how often `title` and `original_title` are identical (case-sensitive and case-insensitive) on the two chosen candidates, then breaks the rate down by `original_language` using the same top-10 pattern as the tagline/overview checks above -- likely near-100% for English-original films and much lower for translated titles.

### Title/original_title equality for `tmdb_budget_revenue_gt0`

In [ ]:
tmdb_budget_revenue_gt0["title_equals_original_cs"] = tmdb_budget_revenue_gt0["title"] == tmdb_budget_revenue_gt0["original_title"]
tmdb_budget_revenue_gt0["title_equals_original_ci"] = (
    tmdb_budget_revenue_gt0["title"].str.lower() == tmdb_budget_revenue_gt0["original_title"].str.lower()
)

cs_rate = tmdb_budget_revenue_gt0["title_equals_original_cs"].mean()
ci_rate = tmdb_budget_revenue_gt0["title_equals_original_ci"].mean()
cs_rate_br_gt0, ci_rate_br_gt0 = cs_rate, ci_rate
print(f"tmdb_budget_revenue_gt0: title == original_title (case-sensitive): {cs_rate:.1%}")
print(f"tmdb_budget_revenue_gt0: title == original_title (case-insensitive): {ci_rate:.1%}")

top10_languages = tmdb_budget_revenue_gt0["original_language"].value_counts().head(10).index
top10_mask = tmdb_budget_revenue_gt0["original_language"].isin(top10_languages)

equality_by_language = (
    tmdb_budget_revenue_gt0[top10_mask]
    .groupby("original_language")[["title_equals_original_cs", "title_equals_original_ci"]]
    .mean()
    .sort_values("title_equals_original_ci")
)
equality_by_language.loc["Overall (all languages)"] = [cs_rate, ci_rate]

print("\ntitle == original_title equality rate by language (top 10 languages by row count):")
display(equality_by_language.round(3))

### Title/original_title equality for `tmdb_no_budget_revenue_gt5`

In [ ]:
tmdb_no_budget_revenue_gt5["title_equals_original_cs"] = tmdb_no_budget_revenue_gt5["title"] == tmdb_no_budget_revenue_gt5["original_title"]
tmdb_no_budget_revenue_gt5["title_equals_original_ci"] = (
    tmdb_no_budget_revenue_gt5["title"].str.lower() == tmdb_no_budget_revenue_gt5["original_title"].str.lower()
)

cs_rate = tmdb_no_budget_revenue_gt5["title_equals_original_cs"].mean()
ci_rate = tmdb_no_budget_revenue_gt5["title_equals_original_ci"].mean()
cs_rate_nbr_gt5, ci_rate_nbr_gt5 = cs_rate, ci_rate
print(f"tmdb_no_budget_revenue_gt5: title == original_title (case-sensitive): {cs_rate:.1%}")
print(f"tmdb_no_budget_revenue_gt5: title == original_title (case-insensitive): {ci_rate:.1%}")

top10_languages = tmdb_no_budget_revenue_gt5["original_language"].value_counts().head(10).index
top10_mask = tmdb_no_budget_revenue_gt5["original_language"].isin(top10_languages)

equality_by_language = (
    tmdb_no_budget_revenue_gt5[top10_mask]
    .groupby("original_language")[["title_equals_original_cs", "title_equals_original_ci"]]
    .mean()
    .sort_values("title_equals_original_ci")
)
equality_by_language.loc["Overall (all languages)"] = [cs_rate, ci_rate]

print("\ntitle == original_title equality rate by language (top 10 languages by row count):")
display(equality_by_language.round(3))

### Drop `_cs` flag, keep `_ci` as `title_differs_from_original`

In [ ]:
tmdb_budget_revenue_gt0["title_differs_from_original"] = (~tmdb_budget_revenue_gt0["title_equals_original_ci"]).astype(bool)
tmdb_budget_revenue_gt0 = tmdb_budget_revenue_gt0.drop(
    columns=["title_equals_original_cs", "title_equals_original_ci"]
)

tmdb_no_budget_revenue_gt5["title_differs_from_original"] = (~tmdb_no_budget_revenue_gt5["title_equals_original_ci"]).astype(bool)
tmdb_no_budget_revenue_gt5 = tmdb_no_budget_revenue_gt5.drop(
    columns=["title_equals_original_cs", "title_equals_original_ci"]
)

In [ ]:
print(
    f"tmdb_budget_revenue_gt0: {tmdb_budget_revenue_gt0['title_differs_from_original'].mean():.1%} "
    "of titles differ from original_title"
)
print(
    f"tmdb_no_budget_revenue_gt5: {tmdb_no_budget_revenue_gt5['title_differs_from_original'].mean():.1%} "
    "of titles differ from original_title"
)

## 7. Drop vote_count and imdb_id

`vote_count` defined the row-selection filter used to build this candidate dataset (`vote_count > threshold`), so by construction every remaining row already satisfies that condition. Keeping it as an input feature would let the model partly learn its own selection criterion rather than anything about the film.

`imdb_id` is an external identifier with no content beyond uniquely naming a row -- it is excluded from every encoder and has only been retained in the DataFrame up to this point for traceability. Both features are dropped here for these reasons.

### Drop `vote_count` and `imdb_id`

In [ ]:
tmdb_budget_revenue_gt0 = tmdb_budget_revenue_gt0.drop(columns=["vote_count", "imdb_id"])
tmdb_no_budget_revenue_gt5 = tmdb_no_budget_revenue_gt5.drop(columns=["vote_count", "imdb_id"])

print(f"tmdb_budget_revenue_gt0: {tmdb_budget_revenue_gt0.shape}")
print(f"tmdb_no_budget_revenue_gt5: {tmdb_no_budget_revenue_gt5.shape}")

## 8. Recode missing list-valued fields as empty lists

`01_overview_cleaning.ipynb` recoded empty lists to NaN for the five list-valued fields (`genres`, `keywords`, `production_companies`, `production_countries`, `spoken_languages`), so missingness reports would flag them consistently with every other feature. This reverses that here, on both chosen candidates, ahead of saving -- restoring an empty list wherever the value is missing, since `pd.read_parquet` deserializes list cells as `numpy.ndarray`, not Python `list`, `pd.api.types.is_list_like` is used instead of `isinstance(x, list)` to check for a real value.

In [ ]:
list_features = ["genres", "keywords", "production_companies", "production_countries", "spoken_languages"]

for col in list_features:
    tmdb_budget_revenue_gt0[col] = tmdb_budget_revenue_gt0[col].apply(
        lambda x: x if pd.api.types.is_list_like(x) else []
    )
    tmdb_no_budget_revenue_gt5[col] = tmdb_no_budget_revenue_gt5[col].apply(
        lambda x: x if pd.api.types.is_list_like(x) else []
    )

print(f"tmdb_budget_revenue_gt0: {tmdb_budget_revenue_gt0[list_features].isnull().sum().sum()} remaining NaN in list-valued fields")
print(f"tmdb_no_budget_revenue_gt5: {tmdb_no_budget_revenue_gt5[list_features].isnull().sum().sum()} remaining NaN in list-valued fields")

## 9. Skewness check and log-transform

Checks the skewness (adjusted Fisher-Pearson standardized moment coefficient, i.e. `pandas`' default `.skew()`) of every numeric column in both chosen candidates, flagging `|skew| > 2` -- the same threshold used in `00_tmdb_eda.ipynb`'s skewness check.

In [ ]:
SKEW_EXCLUDE_COLS = ["id", "release_month_sin", "release_month_cos"]


def skewness_report(df):
    numeric_cols = df.select_dtypes(include="number").columns.difference(SKEW_EXCLUDE_COLS)
    skew = df[numeric_cols].skew()
    return pd.DataFrame(
        {
            "skew": skew,
            "flagged (|skew| > 2)": skew.abs() > 2,
        }
    ).sort_values("skew", key=abs, ascending=False)


br_gt0_skew_before = skewness_report(tmdb_budget_revenue_gt0)
nbr_gt5_skew_before = skewness_report(tmdb_no_budget_revenue_gt5)

print("tmdb_budget_revenue_gt0:")
display(br_gt0_skew_before)

print("tmdb_no_budget_revenue_gt5:")
display(nbr_gt5_skew_before)

### Log-transform `popularity`, `revenue`, and `budget`

The most heavily skewed features per the table above. `popularity` uses `log1p` (not `log`, since `popularity` can be 0) on both chosen candidates. `revenue`/`budget` use plain `log` (both are strictly positive here -- zeros and negatives were already recoded to NaN and dropped upstream) and only exist on `tmdb_budget_revenue_gt0`, since `tmdb_no_budget_revenue_gt5` drops both columns entirely.

In [ ]:
tmdb_budget_revenue_gt0["popularity"] = np.log1p(tmdb_budget_revenue_gt0["popularity"])
tmdb_no_budget_revenue_gt5["popularity"] = np.log1p(tmdb_no_budget_revenue_gt5["popularity"])

In [ ]:
tmdb_budget_revenue_gt0["revenue"] = np.log(tmdb_budget_revenue_gt0["revenue"])
tmdb_budget_revenue_gt0["budget"] = np.log(tmdb_budget_revenue_gt0["budget"])

In [ ]:
br_gt0_skew_after = skewness_report(tmdb_budget_revenue_gt0)
nbr_gt5_skew_after = skewness_report(tmdb_no_budget_revenue_gt5)

print("tmdb_budget_revenue_gt0:")
display(br_gt0_skew_after)

print("tmdb_no_budget_revenue_gt5:")
display(nbr_gt5_skew_after)

## 10. Save chosen datasets

Saves the two chosen candidates to `data/processed/` under short names, for `03_feature_split.ipynb` to load directly.

In [ ]:
tmdb_br_gt0_path = PROCESSED_DIR / "tmdb_br_gt0.parquet"
tmdb_nbr_gt5_path = PROCESSED_DIR / "tmdb_nbr_gt5.parquet"

tmdb_budget_revenue_gt0.to_parquet(tmdb_br_gt0_path, index=False)
tmdb_no_budget_revenue_gt5.to_parquet(tmdb_nbr_gt5_path, index=False)

print(f"Saved: {tmdb_br_gt0_path} ({tmdb_budget_revenue_gt0.shape})")
print(f"Saved: {tmdb_nbr_gt5_path} ({tmdb_no_budget_revenue_gt5.shape})")

## 11. Log summary

In [ ]:
findings = {
    "Base cleaned dataset (from 01)": tmdb.shape,
    "7 candidate shapes": {name: df.shape for name, df in tmdb_candidates.items()},
    "Rows dropped for missing release_date": {
        "tmdb_budget_revenue_gt0": n_dropped_release_date_br_gt0,
        "tmdb_no_budget_revenue_gt5": n_dropped_release_date_nbr_gt5,
    },
    "Chosen candidates, final shape": {
        "tmdb_budget_revenue_gt0": tmdb_budget_revenue_gt0.shape,
        "tmdb_no_budget_revenue_gt5": tmdb_no_budget_revenue_gt5.shape,
    },
    "tagline/overview presence (overall)": {
        "tmdb_budget_revenue_gt0": f"{overall_has_tagline_rate_br_gt0:.1%} / {overall_has_overview_rate_br_gt0:.1%}",
        "tmdb_no_budget_revenue_gt5": f"{overall_has_tagline_rate_nbr_gt5:.1%} / {overall_has_overview_rate_nbr_gt5:.1%}",
    },
    "overview missing values": "filled with \"\" (empty string), not left as NaN (see Section 4)",
    "title == original_title (cs/ci)": {
        "tmdb_budget_revenue_gt0": f"{cs_rate_br_gt0:.1%} / {ci_rate_br_gt0:.1%}",
        "tmdb_no_budget_revenue_gt5": f"{cs_rate_nbr_gt5:.1%} / {ci_rate_nbr_gt5:.1%}",
    },
    "title_differs_from_original (retained feature - case-sensitive)": (
        f"{tmdb_budget_revenue_gt0['title_differs_from_original'].mean():.1%} / "
        f"{tmdb_no_budget_revenue_gt5['title_differs_from_original'].mean():.1%}"
    ),
    "Kruskal-Wallis on release_year by language": {
        "tmdb_budget_revenue_gt0": f"H={h_stat_br_gt0:.2f}, p={p_value_br_gt0:.4g}",
        "tmdb_no_budget_revenue_gt5": f"H={h_stat_nbr_gt5:.2f}, p={p_value_nbr_gt5:.4g}",
    },
    "release_date decomposition": (
        "release_year + release_month_sin/cos extracted; release_date and "
        "raw release_month then dropped (see Section 5)"
    ),
    "Dropped along the way": (
        "tagline, has_overview, vote_count, imdb_id, plus the tagline/overview "
        "presence flags and the cs equality flag (see Sections 2, 3, 6, 7); the "
        "ci equality flag is kept, renamed to title_differs_from_original"
    ),
    "Log-transformed for skew": (
        "popularity (log1p, both candidates); revenue/budget (log, "
        "tmdb_budget_revenue_gt0 only) -- see Section 9 before/after "
        "skewness tables"
    ),
    "Saved to": f"{tmdb_br_gt0_path}, {tmdb_nbr_gt5_path}",
}

log_findings(
    title="02 Preprocessing: TMDB candidates -- summary",
    findings=findings,
)
print("Logged summary to reports/eda_log.md")